# 🔬 Chronos-2 FD002 — Multi-Seed Evaluation (5 seeds)

Clean, reproducible evaluation reporting **mean ± std** over 5 random seeds, using the official
C-MAPSS train/test split (comparable to the literature).

**Design:**
- Embeddings are **deterministic** (Chronos doesn't depend on the seed) → extracted **once per (feature_set × pooling)** and cached to disk.
- Only the **regression heads** are re-trained across seeds → fast, since heads are cheap.
- Reports mean ± std of RMSE and NASA Score for Ridge, MLP, XGBoost.

**Seeds:** 42, 123, 456, 789, 1000
**Author:** Fatima Khadija Benzine

---
## Setup

In [ ]:
# Install dependencies
!pip install -q 'chronos-forecasting>=2.2' xgboost torch

import os
!rm -rf /content/RUL-Chronos
!git clone https://github.com/f-khadija-benzine/RUL-Chronos.git /content/RUL-Chronos

os.chdir('/content/RUL-Chronos/src')

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import json, time, warnings
warnings.filterwarnings('ignore')

import torch
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error
from xgboost import XGBRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

from data_loader import MultiDatasetLoader
from preprocessing import DataNormalizer

TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M')
SAVE_DIR = f'/content/drive/MyDrive/PhD_results/Chronos_Eval_{TIMESTAMP}'
os.makedirs(SAVE_DIR, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
print(f"Save directory: {SAVE_DIR}")
print("Setup complete ✓")

---
## Part A — Data Preparation (FD002, per-regime)

In [ ]:
# === A1: Load C-MAPSS FD002 ===
DATASET = 'FD002'          # 6 operating conditions
W = 30
RUL_CAP = 125
N_REGIMES = 6              # FD002 has 6 operating regimes

loader = MultiDatasetLoader()
ds = loader.load_cmapss_dataset(DATASET)

train_raw = ds['train'].copy()
test_raw = ds['test'].copy()

# Apply RUL cap
train_raw['rul'] = train_raw['rul'].clip(upper=RUL_CAP)
if 'rul' in test_raw.columns:
    test_raw['rul'] = test_raw['rul'].clip(upper=RUL_CAP)

sensor_cols = [c for c in train_raw.columns if c.startswith('sensor_')]
setting_cols = [c for c in train_raw.columns if c.startswith('setting_')]

print(f"✓ Data loaded: {DATASET}")
print(f"  Train: {len(train_raw)} samples, {train_raw['unit'].nunique()} units")
print(f"  Test:  {len(test_raw)} samples, {test_raw['unit'].nunique()} units")
print(f"  Sensors: {len(sensor_cols)}, Settings: {len(setting_cols)}")

In [ ]:
# === A2a: Identify the 6 Operating Regimes ===
# Cluster the 3 operating settings into 6 groups. Fit on TRAIN only,
# then assign the same clusters to TEST (no leakage).

kmeans = KMeans(n_clusters=N_REGIMES, random_state=42, n_init=10)
train_raw['regime'] = kmeans.fit_predict(train_raw[setting_cols].values)
test_raw['regime']  = kmeans.predict(test_raw[setting_cols].values)

print(f"✓ Identified {N_REGIMES} operating regimes")
print("\nRegime distribution (train):")
print(train_raw['regime'].value_counts().sort_index().to_string())
print("\nRegime distribution (test):")
print(test_raw['regime'].value_counts().sort_index().to_string())

In [ ]:
# === A2b: Visualize the 6 Operating Regimes ===
# Confirm the operating settings separate cleanly into 6 clusters.

fig = plt.figure(figsize=(14, 4))
colors = plt.cm.tab10(np.linspace(0, 1, N_REGIMES))

# 3D-style view via three 2D projections of the settings
pairs = [(0, 1), (0, 2), (1, 2)]
for idx, (a, b) in enumerate(pairs):
    ax = fig.add_subplot(1, 3, idx + 1)
    for r in range(N_REGIMES):
        mask = train_raw['regime'] == r
        ax.scatter(train_raw.loc[mask, setting_cols[a]],
                   train_raw.loc[mask, setting_cols[b]],
                   s=6, alpha=0.4, color=colors[r], label=f'Regime {r}')
    ax.set_xlabel(setting_cols[a])
    ax.set_ylabel(setting_cols[b])
    ax.set_title(f'{setting_cols[a]} vs {setting_cols[b]}')
    if idx == 2:
        ax.legend(markerscale=2, fontsize=8, loc='center left',
                  bbox_to_anchor=(1.02, 0.5))

plt.suptitle('FD002 Operating Regimes (K-Means, k=6)', fontweight='bold')
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fd002_regimes_{TIMESTAMP}.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ The 6 regimes should appear as clearly separated clusters.")

In [ ]:
# === A2c: Per-Regime (Condition-Specific) Normalization ===
# Fit one MinMax scaler PER regime on sensors, using TRAIN only,
# then apply to both train and test. This removes the regime effect
# so the normalized signal reflects degradation, not operating condition.

scalers = {}
train_norm = train_raw.copy()
test_norm  = test_raw.copy()

for r in range(N_REGIMES):
    scaler = MinMaxScaler()
    tr_mask = train_raw['regime'] == r
    scaler.fit(train_raw.loc[tr_mask, sensor_cols].values)
    scalers[r] = scaler

    train_norm.loc[tr_mask, sensor_cols] = scaler.transform(
        train_raw.loc[tr_mask, sensor_cols].values)

    te_mask = test_raw['regime'] == r
    if te_mask.any():
        test_norm.loc[te_mask, sensor_cols] = scaler.transform(
            test_raw.loc[te_mask, sensor_cols].values)

# After per-regime normalization, sensors are the feature pool.
# (Settings only encoded the regime, which is now normalized out.)
feature_cols = sensor_cols

print("✓ Per-regime MinMax normalization complete")
print(f"  Feature pool: {len(feature_cols)} sensors")

# Sanity check: a sensor should now look like a degradation trend,
# not a step function jumping between regimes.
u = sorted(train_norm['unit'].unique())[0]
ud = train_norm[train_norm['unit'] == u].sort_values('cycle')
plt.figure(figsize=(12, 3))
for s in sensor_cols[:4]:
    plt.plot(ud['cycle'], ud[s], label=s, alpha=0.8)
plt.xlabel('Cycle'); plt.ylabel('Normalized value')
plt.title(f'Normalized sensors for unit {u} (should show smooth degradation trends)')
plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/fd002_normcheck_{TIMESTAMP}.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# === A3: Feature Selection ===

def variance_filter(df, feature_cols, threshold=0.001):
    """Remove low-variance features"""
    variances = df[feature_cols].var()
    selected = variances[variances > threshold].index.tolist()
    removed = [f for f in feature_cols if f not in selected]
    return selected, removed

def correlation_filter(df, feature_cols, threshold=0.95):
    """Remove highly correlated features"""
    corr_matrix = df[feature_cols].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > threshold)]
    selected = [f for f in feature_cols if f not in to_drop]
    return selected, to_drop

def aficv_selection(df, feature_cols, target_col='rul', threshold=0.90):
    """AFICv: Accumulated Feature Importance with Cross-Validation"""
    X = df[feature_cols].values
    y = df[target_col].values

    model = XGBRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1)
    model.fit(X, y)

    importances = model.feature_importances_
    indices = np.argsort(importances)[::-1]

    cumsum = np.cumsum(importances[indices])
    n_features = np.searchsorted(cumsum, threshold) + 1

    selected_indices = indices[:n_features]
    selected = [feature_cols[i] for i in selected_indices]

    return selected, cumsum[n_features-1]

# Strategy 1: Variance + Correlation filtering
print("=== Strategy 1: Variance + Correlation Filtering ===")
feats_var, removed_var = variance_filter(train_norm, feature_cols, threshold=0.001)
print(f"  Variance filter: {len(feature_cols)} → {len(feats_var)} (removed: {removed_var})")

feats_corr, removed_corr = correlation_filter(train_norm, feats_var, threshold=0.95)
print(f"  Correlation filter: {len(feats_var)} → {len(feats_corr)} (removed: {removed_corr})")

# Strategy 2: AFICv
print("\n=== Strategy 2: AFICv (90% importance coverage) ===")
feats_aficv, coverage = aficv_selection(train_norm, feature_cols, threshold=0.90)
print(f"  Selected: {len(feats_aficv)} features (coverage: {coverage:.1%})")
print(f"  Features: {feats_aficv}")

# Store feature sets
feature_sets = {
    'correlation': feats_corr,
    'aficv': feats_aficv
}

print("\n=== Summary ===")
for name, feats in feature_sets.items():
    print(f"  {name}: {len(feats)} features")

In [ ]:
# === A4: Create Sliding Windows ===

def create_sliding_windows(df, feature_cols, window_size=30):
    """Create sliding windows for each unit"""
    X_list, y_list, unit_list = [], [], []

    for unit in sorted(df['unit'].unique()):
        unit_data = df[df['unit'] == unit].sort_values('cycle')
        n_samples = len(unit_data)

        if n_samples < window_size:
            continue

        values = unit_data[feature_cols].values
        ruls = unit_data['rul'].values

        for start in range(n_samples - window_size + 1):
            end = start + window_size
            X_list.append(values[start:end])
            y_list.append(ruls[end - 1])
            unit_list.append(unit)

    return np.array(X_list), np.array(y_list), np.array(unit_list)

# Create windows for each feature set
data_dict = {}

for fs_name, features in feature_sets.items():
    X_train, y_train, units_train = create_sliding_windows(train_norm, features, W)
    X_test, y_test, units_test = create_sliding_windows(test_norm, features, W)

    data_dict[fs_name] = {
        'X_train': X_train, 'y_train': y_train, 'units_train': units_train,
        'X_test': X_test, 'y_test': y_test, 'units_test': units_test,
        'features': features
    }

    print(f"{fs_name}: X_train={X_train.shape}, X_test={X_test.shape}")

print("\n✓ Sliding windows created")

In [ ]:
# === A5: Evaluation Helpers ===

def nasa_score(y_true, y_pred):
    """NASA asymmetric scoring function (Saxena et al. 2008)"""
    d = y_pred - y_true
    score = np.where(d < 0, np.exp(-d/13) - 1, np.exp(d/10) - 1)
    return np.sum(score)

def evaluate_per_unit(y_true, y_pred, unit_labels):
    """Evaluate using last prediction per unit (standard C-MAPSS protocol)"""
    preds_last, true_last = [], []
    for u in sorted(set(unit_labels)):
        mask = unit_labels == u
        if mask.sum() > 0:
            preds_last.append(y_pred[mask][-1])
            true_last.append(y_true[mask][-1])

    preds_last = np.array(preds_last)
    true_last = np.array(true_last)

    rmse = np.sqrt(mean_squared_error(true_last, preds_last))
    score = nasa_score(true_last, preds_last)
    return rmse, score

ALL_RESULTS = []
print("Evaluation helpers ✓")

---
## Part B — Load Chronos-2

In [ ]:
# === C1: Load Chronos-2 Pipeline ===
from chronos import Chronos2Pipeline

CHRONOS2_MODEL = "amazon/chronos-2"

print(f"Loading Chronos-2: {CHRONOS2_MODEL}...")
chronos2_pipeline = Chronos2Pipeline.from_pretrained(
    CHRONOS2_MODEL,
    device_map=DEVICE
)
print("Chronos-2 loaded ✓")
print(f"  Parameters: 120M")
print(f"  Architecture: Encoder-only + Group Attention")
print(f"  Capabilities: Univariate, Multivariate")

In [ ]:
# === C2: Extract Chronos-2 Embeddings (Multivariate) ===

def extract_chronos2_embeddings(pipeline, X, batch_size=32):
    """
    Extract Chronos-2 encoder embeddings.
    Chronos-2 can process multivariate series natively.
    X shape: (n_samples, window_size, n_features)
    """
    n_samples, window_size, n_features = X.shape
    all_embeddings = []

    for i in range(0, n_samples, batch_size):
        batch = X[i:i+batch_size]
        batch_t = np.transpose(batch, (0, 2, 1))

        ts_tensor = torch.tensor(batch_t, dtype=torch.float32)

        with torch.no_grad():
            result = pipeline.embed(ts_tensor)
            # embed() returns (embeddings, loc_scale) tuple
            emb = result[0] if isinstance(result, tuple) else result
            emb_pooled = emb.mean(dim=1).cpu().numpy()

        all_embeddings.append(emb_pooled)

        if (i // batch_size) % 10 == 0:
            print(f"    Processed {min(i+batch_size, n_samples)}/{n_samples} samples")

    return np.vstack(all_embeddings)

print("Chronos-2 embedding function defined ✓")

In [ ]:
# === C2b: Different Embedding Pooling Strategies ===

def extract_chronos2_embeddings_v2(pipeline, X, batch_size=32, pooling='mean'):
    """
    Extract Chronos-2 embeddings with different pooling strategies.

    Args:
        pooling: 'mean' | 'last' | 'concat'
            - mean: Average over features and patches (current approach)
            - last: Take only the last patch embedding
            - concat: Concatenate all patch embeddings (larger dim)
    """
    n_samples, window_size, n_features = X.shape
    all_embeddings = []

    for i in range(0, n_samples, batch_size):
        batch = X[i:i+batch_size]
        batch_t = np.transpose(batch, (0, 2, 1))  # (batch, features, window)
        ts_tensor = torch.tensor(batch_t, dtype=torch.float32)

        with torch.no_grad():
            result = pipeline.embed(ts_tensor)
            emb_list = result[0]  # list of tensors
            emb = torch.stack(emb_list)  # (batch, n_features, n_patches, embed_dim)

            if pooling == 'mean':
                # Current: mean over features and patches -> (batch, embed_dim)
                emb_pooled = emb.mean(dim=(1, 2)).cpu().numpy()

            elif pooling == 'last':
                # Take last patch only, mean over features -> (batch, embed_dim)
                emb_pooled = emb[:, :, -1, :].mean(dim=1).cpu().numpy()

            elif pooling == 'concat':
                # Concatenate all patches, mean over features -> (batch, n_patches * embed_dim)
                batch_size_actual = emb.shape[0]
                n_patches = emb.shape[2]
                embed_dim = emb.shape[3]
                # Mean over features first -> (batch, n_patches, embed_dim)
                emb_feat_mean = emb.mean(dim=1)
                # Flatten patches -> (batch, n_patches * embed_dim)
                emb_pooled = emb_feat_mean.reshape(batch_size_actual, -1).cpu().numpy()

            elif pooling == 'last_concat':
                # Concatenate last patch from all features -> (batch, n_features * embed_dim)
                emb_pooled = emb[:, :, -1, :].reshape(emb.shape[0], -1).cpu().numpy()

        all_embeddings.append(emb_pooled)

    return np.vstack(all_embeddings)

print("Pooling strategies defined:")
print("  - mean: (batch, 768) - current approach")
print("  - last: (batch, 768) - last patch only")
print("  - concat: (batch, n_patches * 768) - all patches concatenated")
print("  - last_concat: (batch, n_features * 768) - last patch per feature")

---
## Part C — Multi-Seed Engine

The core idea: **extract embeddings once (cached), loop seeds only over the heads.**

In [ ]:
# === Multi-seed configuration ===
SEEDS = [42, 123, 456, 789, 1000]

# Which pooling strategies to evaluate (add/remove as needed)
POOLINGS = ['mean', 'last', 'concat', 'last_concat']

# Fixed cache dir for embeddings (survives restarts; independent of seed)
EMB_ROOT = '/content/drive/MyDrive/PhD_results/embeddings_cache/FD002/chronos2_pool'
os.makedirs(EMB_ROOT, exist_ok=True)

print(f"Seeds: {SEEDS}")
print(f"Poolings: {POOLINGS}")
print(f"Embedding cache: {EMB_ROOT}")

In [ ]:
# === Cached embedding extraction (once per feature_set × pooling) ===
def get_pooled_embeddings(fs_name, pooling):
    """Return (X_train_emb, X_test_emb), computing+caching if needed.
    Embeddings do NOT depend on the seed, so we cache them."""
    data = data_dict[fs_name]
    tr_path = os.path.join(EMB_ROOT, f'{fs_name}_{pooling}_train.npy')
    te_path = os.path.join(EMB_ROOT, f'{fs_name}_{pooling}_test.npy')

    if os.path.exists(tr_path) and os.path.exists(te_path):
        return np.load(tr_path), np.load(te_path)

    print(f"    computing embeddings [{fs_name} | {pooling}] ...")
    Xtr = extract_chronos2_embeddings_v2(chronos2_pipeline, data['X_train'], pooling=pooling)
    Xte = extract_chronos2_embeddings_v2(chronos2_pipeline, data['X_test'],  pooling=pooling)
    np.save(tr_path, Xtr)
    np.save(te_path, Xte)
    print(f"    cached  train={Xtr.shape}  test={Xte.shape}")
    return Xtr, Xte

print("Embedding cache helper ready ✓")

In [ ]:
# === Multi-seed head training ===
def make_heads(seed):
    """Fresh heads for a given seed. Ridge is deterministic (seed has no effect)."""
    return [
        ('Ridge',   Ridge(alpha=1.0)),
        ('MLP',     MLPRegressor(hidden_layer_sizes=(256,128), max_iter=500,
                                 early_stopping=True, n_iter_no_change=10,
                                 random_state=seed)),
        ('XGBoost', XGBRegressor(n_estimators=100, max_depth=5, random_state=seed)),
    ]

def run_multiseed(fs_name, pooling):
    """Train every head across all seeds on cached embeddings.
    Returns a list of per-(head) aggregated dicts (mean±std)."""
    data = data_dict[fs_name]
    Xtr, Xte = get_pooled_embeddings(fs_name, pooling)

    # collect raw metrics: {head: {'rmse': [...], 'score': [...]}}
    raw = {}
    for seed in SEEDS:
        for head_name, head_model in make_heads(seed):
            head_model.fit(Xtr, data['y_train'])
            y_pred = head_model.predict(Xte)
            rmse, score = evaluate_per_unit(data['y_test'], y_pred, data['units_test'])
            raw.setdefault(head_name, {'rmse': [], 'score': []})
            raw[head_name]['rmse'].append(rmse)
            raw[head_name]['score'].append(score)
        # Ridge is deterministic → no need to repeat across seeds
        # (we still stored it each seed; std will be ~0, which is correct/expected)

    rows = []
    for head_name, m in raw.items():
        rmse_arr, score_arr = np.array(m['rmse']), np.array(m['score'])
        rows.append({
            'Feature_Set': fs_name,
            'Pooling': pooling,
            'Head': head_name,
            'Embed_Dim': Xtr.shape[1],
            'RMSE_mean': round(rmse_arr.mean(), 2),
            'RMSE_std':  round(rmse_arr.std(), 2),
            'Score_mean': round(score_arr.mean(), 1),
            'Score_std':  round(score_arr.std(), 1),
            'n_seeds': len(SEEDS),
        })
    return rows

print("Multi-seed runner ready ✓")

---
## Part D — Run the Grid

In [ ]:
# === Run: all feature sets × all poolings × all heads × 5 seeds ===
MULTISEED_RESULTS = []

for fs_name in feature_sets.keys():
    for pooling in POOLINGS:
        print(f"\n{'='*60}\n{fs_name.upper()} | pooling={pooling}\n{'='*60}")
        rows = run_multiseed(fs_name, pooling)
        MULTISEED_RESULTS.extend(rows)
        for r in rows:
            print(f"  {r['Head']:8s}  RMSE {r['RMSE_mean']:.2f} ± {r['RMSE_std']:.2f}"
                  f"   Score {r['Score_mean']:.1f} ± {r['Score_std']:.1f}")

print("\n✓ Grid complete")

In [ ]:
# === Results table (sorted by mean RMSE) ===
ms_df = pd.DataFrame(MULTISEED_RESULTS)
ms_df = ms_df.sort_values(['Feature_Set', 'RMSE_mean'])
# pretty display column: "mean ± std"
ms_df['RMSE'] = ms_df.apply(lambda r: f"{r['RMSE_mean']:.2f} ± {r['RMSE_std']:.2f}", axis=1)
ms_df['Score'] = ms_df.apply(lambda r: f"{r['Score_mean']:.1f} ± {r['Score_std']:.1f}", axis=1)
display_cols = ['Feature_Set','Pooling','Head','Embed_Dim','RMSE','Score','n_seeds']
print(ms_df[display_cols].to_string(index=False))

---
## Part E — Save to `results/offline/`

In [ ]:
# === Save multi-seed results ===
OFFLINE_DIR = '/content/drive/MyDrive/PhD_results/offline'
os.makedirs(OFFLINE_DIR, exist_ok=True)

csv_path  = os.path.join(OFFLINE_DIR, 'fd002_multiseed_pooling.csv')
json_path = os.path.join(OFFLINE_DIR, 'fd002_multiseed_pooling.json')

ms_df[display_cols + ['RMSE_mean','RMSE_std','Score_mean','Score_std']].to_csv(csv_path, index=False)
with open(json_path, 'w') as f:
    json.dump(MULTISEED_RESULTS, f, indent=2)

print(f"✓ Saved {csv_path}")
print(f"✓ Saved {json_path}")
print("\nCommit only this folder:  git add results/offline/")